# Partie 1.1 : analyse exploratoire et découpage des données

Ce notebook prépare les données une seule fois pour tout le projet :

- il compte les images de chaque classe et en affiche des exemples ;
- il cherche les images en double (la même feuille tournée ou retournée) pour éviter les fuites de données ;
- il fait le découpage 70 % / 15 % / 15 % (stratifié, graine 42) et l'enregistre dans `data/splits.csv` ;
- il calcule les statistiques de normalisation sur le train ;
- il prépare le jeu hors-distribution de la partie 4.

Tous les autres notebooks relisent `data/splits.csv`. Il ne faut plus le régénérer une fois les entraînements commencés.

- On ajoute le dossier du projet au chemin Python pour importer notre code (`src/`).
- `config` contient les chemins, la classe M choisie et le GPU utilisé.
- `set_seed(42)` fixe le hasard : on obtient les mêmes résultats à chaque exécution.
- La dernière ligne affiche la machine utilisée (à noter dans le rapport).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # dossier du projet, pour pouvoir importer src/

from src import config as C
C.set_seed(42)
print(C.describe_env())

- Imports des bibliothèques et de nos fonctions (`data`, `dedup` pour les doublons, `viz` pour les grilles d'images).
- `SEUIL = 4` : deux images dont les hash diffèrent de 4 bits ou moins (sur 64) sont considérées comme la même feuille. On vérifie ce choix plus bas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hashlib, shutil
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

from src import data as D, dedup, viz
from src.metrics import savefig

SEUIL = 4   # nombre de bits maximum entre deux hash pour dire que c'est la même feuille

## Les données

- `ensure_raw_data()` télécharge PlantVillage depuis Kaggle et copie les 10 dossiers de tomate dans `data/raw/`.
- On prend la version originale « color » et pas la version « Augmented » de Kaggle, qui contient déjà des copies tournées des mêmes feuilles.
- On liste toutes les images avec leur classe. Le chemin est gardé en relatif (`raw/...`) pour que le fichier marche sur n'importe quel ordinateur.
- On affiche le nombre d'images par classe : c'est le chiffre exact à mettre dans le rapport.

In [ ]:
color_dir = D.ensure_raw_data()
print("Source : Kaggle", C.KAGGLE_DATASET, "(PlantVillage original, dossier color)")

lignes = []
for classe in D.CLASSES:
    for fichier in D.list_images(C.RAW_DIR / classe):
        lignes.append({"path": f"raw/{classe}/{fichier.name}", "class": classe})
df = pd.DataFrame(lignes).sort_values(["class", "path"]).reset_index(drop=True)

effectifs = df["class"].value_counts().reindex(D.CLASSES)
print("Nombre total d'images :", len(df))
effectifs.rename(index=D.short).to_frame("nombre d'images")

- Diagramme en barres du nombre d'images par classe.
- Le rapport entre la plus grande et la plus petite classe montre le déséquilibre. Il justifie l'usage du F1 macro (voir plus bas).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
noms = [D.short(c) for c in D.CLASSES]
ax.barh(noms, effectifs.values)
for i, v in enumerate(effectifs.values):
    ax.text(v + 30, i, str(v), va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Nombre d'images")
ax.set_title(f"Nombre d'images par classe (total : {len(df)})")
savefig("00_effectifs.png")
plt.show()
print(f"La plus grande classe a {effectifs.max() / effectifs.min():.1f} fois plus d'images que la plus petite.")

- On tire 5 images au hasard dans chaque classe et on les affiche.
- On remarque que toutes les photos ont le même fond uni de laboratoire. On y reviendra en partie 6 (biais de fond).

In [ ]:
images, titres = [], []
for classe in D.CLASSES:
    exemples = df[df["class"] == classe].sample(5, random_state=42)
    for chemin in exemples["path"]:
        images.append(D.read_rgb(C.resolve(chemin), 128))
        titres.append(D.short(classe))
viz.image_grid(images, titres, ncols=10, size=1.5, path=C.FIG_DIR / "00_exemples.png",
               suptitle="5 exemples par classe")

## Vérification anti-fuite : recherche des doublons

Certaines versions de PlantVillage contiennent la même feuille plusieurs fois (tournée ou retournée). Si une copie est dans le train et une autre dans le test, le test devient trop facile : c'est une fuite de données.

Méthode :

1. pour chaque image on calcule un **hash perceptuel** (pHash), une empreinte de 64 bits qui change peu si l'image est compressée ou redimensionnée ;
2. on le calcule aussi pour les 8 versions de l'image (4 rotations, avec et sans miroir) ;
3. deux images sont la même feuille si leurs empreintes diffèrent de 4 bits au plus ;
4. les images qui sont la même feuille forment un **groupe** (`leaf_group`), et un groupe ira entièrement dans un seul split.

- On calcule les 8 hash de chaque image, puis on les sauvegarde pour ne pas les recalculer la prochaine fois (c'est l'étape la plus longue).
- Le md5 détecte en plus les fichiers copiés à l'identique.

In [ ]:
fichier_hash = C.DATA_DIR / f"phash8_{len(df)}.npy"
if fichier_hash.exists():
    H = np.load(fichier_hash)
else:
    H = dedup.compute_hashes([C.resolve(p) for p in df["path"]])
    np.save(fichier_hash, H)
print("Hash calculés :", H.shape, "(une ligne par image, 8 versions par image)")

df["md5"] = [hashlib.md5(open(C.resolve(p), "rb").read()).hexdigest() for p in df["path"]]
print("Fichiers strictement identiques :", len(df) - df["md5"].nunique())

- On cherche toutes les paires d'images proches (jusqu'à 8 bits d'écart).
- On compte combien de paires on trouverait avec chaque seuil de 0 à 8.
- Si le nombre augmente très vite après 4, c'est qu'on commence à attraper des feuilles différentes mais qui se ressemblent. Ce tableau sert à justifier le seuil de 4.
- `paires` garde seulement les paires au seuil choisi.

In [ ]:
paires_8 = dedup.find_pairs(H, threshold=8)
distances = np.array([d for _, _, d in paires_8])
print("Nombre de paires selon le seuil :")
for s in range(9):
    print(f"  distance <= {s} bits : {(distances <= s).sum()} paires")
paires = [p for p in paires_8 if p[2] <= SEUIL]

- On affiche 20 paires au hasard, côte à côte.
- Vérification à l'oeil : les paires de même classe doivent être la même feuille (sinon baisser `SEUIL`). Les paires entre deux classes différentes peuvent être de simples ressemblances (voir la cellule suivante).

In [ ]:
ordre = np.random.default_rng(42).permutation(len(paires))[:20]
images, titres = [], []
for k in ordre:
    i, j, d = paires[k]
    images += [D.read_rgb(C.resolve(df.path[i]), 128), D.read_rgb(C.resolve(df.path[j]), 128)]
    titres += [f"{D.short(df['class'][i])} (d={d})", D.short(df["class"][j])]
if images:
    viz.image_grid(images, titres, ncols=8, size=1.6, path=C.FIG_DIR / "00_paires_doublons.png",
                   suptitle="20 paires détectées comme doublons (à vérifier à l'oeil)")

- On regroupe les doublons : si A ressemble à B et B à C, alors A, B et C sont dans le même groupe.
- On affiche le résumé des doublons (à mettre dans le rapport).
- On sépare les paires de même classe et les paires entre deux classes différentes. En regardant la grille, les paires de même classe sont bien la même feuille. Les paires entre deux classes sont en général des feuilles différentes qui ont la même silhouette sur le fond gris : ce sont des fausses alertes du pHash.
- Ces fausses alertes ne posent pas de problème pour la fuite : elles mettent juste deux feuilles différentes dans le même split, ce qui est plus prudent.
- Un groupe énorme voudrait dire que le seuil relie des feuilles différentes en chaîne.

In [ ]:
df["leaf_group"] = dedup.groups_from_pairs(len(df), paires)
taille_groupes = df.groupby("leaf_group").size()
nb_classes_par_groupe = df.groupby("leaf_group")["class"].nunique()

print("Paires de doublons trouvées :", len(paires))
print("Groupes de plus d'une image :", (taille_groupes > 1).sum())
print("Images concernées :", taille_groupes[taille_groupes > 1].sum())
print("Taille du plus gros groupe :", taille_groupes.max())
meme_classe = sum(df["class"][i] == df["class"][j] for i, j, _ in paires)
print(f"Paires de même classe : {meme_classe}, paires entre deux classes différentes : {len(paires) - meme_classe}")
print("Groupes qui mélangent deux classes :", (nb_classes_par_groupe > 1).sum())
if taille_groupes.max() > 50:
    print("ATTENTION : groupe très gros, le seuil est sans doute trop haut.")

## Découpage 70 / 15 / 15

`StratifiedGroupKFold` coupe les données en 20 parts d'environ 5 %. Il respecte deux règles : chaque groupe reste dans une seule part, et chaque part garde les mêmes proportions de classes (stratification). On prend 3 parts pour le test (15 %), 3 pour la validation (15 %) et 14 pour le train (70 %).

- On numérote chaque part de 0 à 19, puis : parts 0 à 2 vers le test, 3 à 5 vers la validation, le reste vers le train.
- On vérifie qu'aucun groupe n'est coupé entre deux splits (sinon le programme s'arrête).
- Le tableau montre, pour chaque classe, la part en train, val et test : chaque ligne doit être proche de 0,70 / 0,15 / 0,15.

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
part = np.empty(len(df), dtype=int)
for k, (_, indices) in enumerate(sgkf.split(df, df["class"], groups=df["leaf_group"])):
    part[indices] = k
df["split"] = np.where(part < 3, "test", np.where(part < 6, "val", "train"))

groupes_partages = (df.groupby("leaf_group")["split"].nunique() > 1).sum()
print("Groupes présents dans plusieurs splits :", groupes_partages)
assert groupes_partages == 0

print(df["split"].value_counts(normalize=True).round(3))
proportions = pd.crosstab(df["class"].map(D.short), df["split"], normalize="index")[["train", "val", "test"]]
proportions.round(3)

- Contre-vérification : pour chaque image du test, on la compare à toutes les versions (rotations, miroirs) des images du train. Même chose pour val/train et test/val.
- On doit trouver 0 image proche et 0 fichier identique. C'est la preuve anti-fuite demandée.

In [ ]:
idx = {s: np.where(df["split"] == s)[0] for s in ["train", "val", "test"]}
for a, b in [("test", "train"), ("val", "train"), ("test", "val")]:
    proches = (dedup.min_distances(H[idx[a]], H[idx[b]]) <= SEUIL).sum()
    md5_communs = len(set(df.md5[idx[a]]) & set(df.md5[idx[b]]))
    print(f"{a} / {b} : {proches} images proches, {md5_communs} fichiers identiques")
    assert proches == 0 and md5_communs == 0
print("Résultat : 0 hash commun entre train, validation et test.")

- On enregistre le découpage dans `data/splits.csv` (colonnes : chemin, classe, groupe, split).

In [ ]:
df[["path", "class", "leaf_group", "split"]].to_csv(C.SPLITS_CSV, index=False)
print("Fichier écrit :", C.SPLITS_CSV)

## Normalisation

- Pour les modèles entraînés **de zéro**, on calcule la moyenne et l'écart-type de chaque canal (R, G, B) **sur le train seulement**. Si on utilisait aussi la validation ou le test, de l'information de ces images entrerait dans l'entraînement.
- Pour le **transfer learning**, on garde les valeurs d'ImageNet, car le modèle pré-entraîné a appris avec elles.
- Pour les **auto-encodeurs**, pas de normalisation : les pixels restent entre 0 et 1.

- On charge les images du train en 128×128.
- On calcule par paquets de 512 images la somme et la somme des carrés des pixels, pour obtenir moyenne et écart-type sans saturer la mémoire.
- On enregistre le résultat dans `results/norm_stats.json`, relu par le notebook 01.
- On compare avec les valeurs ImageNet : les moyennes sont proches, mais les écarts-types de nos images sont plus petits (fond uni, couleurs peu variées). C'est une raison de plus d'utiliser nos propres statistiques pour les modèles de zéro.

In [ ]:
train = df[df["split"] == "train"]
images_train = D.load_cache([C.resolve(p) for p in train["path"]], C.SCRATCH_SIZE)

somme, somme_carres, n = np.zeros(3), np.zeros(3), 0
for k in range(0, len(images_train), 512):
    x = np.asarray(images_train[k:k + 512], dtype=np.float64) / 255
    somme += x.sum(axis=(0, 1, 2))
    somme_carres += (x ** 2).sum(axis=(0, 1, 2))
    n += x.shape[0] * x.shape[1] * x.shape[2]
moyenne = somme / n
ecart_type = np.sqrt(somme_carres / n - moyenne ** 2)

C.save_json({"mean": moyenne.round(5).tolist(), "std": ecart_type.round(5).tolist()}, C.NORM_JSON)
print("Moyenne (R, G, B)    :", moyenne.round(4), " ImageNet :", C.IMAGENET_MEAN)
print("Écart-type (R, G, B) :", ecart_type.round(4), " ImageNet :", C.IMAGENET_STD)

## Déséquilibre des classes

Les classes n'ont pas le même nombre d'images. Un modèle qui ignore la classe rare (mosaïque) perdrait très peu d'accuracy. On fait donc deux choses :

- une perte pondérée : chaque classe reçoit un poids $w_c = \frac{N}{K \times n_c}$ ($N$ images, $K$ classes, $n_c$ images de la classe), donc une erreur sur une classe rare coûte plus cher ;
- le **F1 macro** comme métrique principale : c'est la moyenne des F1 de chaque classe, toutes les classes comptent pareil.

- Calcul des poids de chaque classe à partir du train. Les notebooks d'entraînement refont exactement le même calcul.

In [ ]:
y_train = train["class"].map({c: i for i, c in enumerate(D.CLASSES)}).values
poids = D.class_weights(y_train, len(D.CLASSES))
pd.DataFrame({"images en train": np.bincount(y_train), "poids": poids.numpy().round(3)},
             index=[D.short(c) for c in D.CLASSES])

## Jeu hors-distribution (pour la partie 4)

- **Autres espèces de PlantVillage** (pomme, vigne, pomme de terre...) : on en prend environ 600. On sépare les espèces en deux moitiés : `calib` sert à régler le système hybride (ce ne sont pas des images du test), `eval` sert à l'évaluer.
- **Nos photos** (au moins 30) à mettre dans `data/ood/photos/`, dans les sous-dossiers `tomate_fond_naturel`, `autres_plantes`, `mains`, `sol_objets`.

- On liste les dossiers qui ne sont pas de la tomate et on tire au hasard la moitié des espèces pour `calib`.
- On copie environ 600 images de ces espèces dans `data/ood/plantvillage_autres/`.
- On ajoute nos photos personnelles (la catégorie est le nom du sous-dossier).
- Tout est enregistré dans `data/ood.csv`.
- Cette cellule peut être relancée seule quand les photos sont ajoutées : elle ne touche pas au découpage.

In [ ]:
autres = sorted(d.name for d in color_dir.iterdir() if d.is_dir() and not d.name.startswith("Tomato"))
especes = sorted({a.split("___")[0] for a in autres})
rng = np.random.default_rng(42)
especes_calib = set(rng.choice(especes, size=len(especes) // 2, replace=False))
par_classe = max(1, 600 // max(1, len(autres)))

lignes = []
for a in autres:
    fichiers = D.list_images(color_dir / a)
    for f in rng.choice(fichiers, size=min(par_classe, len(fichiers)), replace=False):
        destination = C.OOD_DIR / "plantvillage_autres" / a / f.name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            shutil.copy(f, destination)
        espece = a.split("___")[0]
        lignes.append({"path": f"ood/plantvillage_autres/{a}/{f.name}", "source": "plantvillage_autres",
                       "category": espece, "ood_split": "calib" if espece in especes_calib else "eval"})

for f in D.list_images(C.OOD_DIR / "photos"):
    lignes.append({"path": "ood/" + f.relative_to(C.OOD_DIR).as_posix(), "source": "photos",
                   "category": f.parent.name, "ood_split": "eval"})

ood = pd.DataFrame(lignes)
ood.to_csv(C.OOD_CSV, index=False)
nb_photos = (ood["source"] == "photos").sum()
print("Espèces pour la calibration :", sorted(map(str, especes_calib)))
print(f"{len(ood)} images hors-distribution, dont {nb_photos} photos personnelles")
if nb_photos < 30:
    print("Il manque des photos : les ajouter dans data/ood/photos/ puis relancer seulement cette cellule.")